# Библиотеки

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

## Преобразование BGR -> RGB

In [2]:
def bgr2rgb(img):
    if len(img.shape) != 3 or img.shape[2] != 3:
        raise ValueError("Требуется цветное изображение (H,W,3)")
    rgb_img = img.copy()
    rgb_img[:,:,0], rgb_img[:,:,2] = img[:,:,2].copy(), img[:,:,0].copy()
    return rgb_img


# ФИЛЬТРЫ

## Медианный фильтр

In [3]:
def median_filter(img, kernel_size=3):
    """Медианный фильтр (блюр)"""
    if kernel_size % 2 == 0:
        raise ValueError("kernel_size должен быть нечётным")
#Вычисляет размер дополнения (padding). Для kernel_size=3 получаем pad=1, что означает добавление по 1 пикселю с каждой стороны.
    pad = kernel_size // 2

#Извлекает размеры изображения: rows — высота, cols — ширина, channels — количество каналов (3 для RGB).
    rows, cols, channels = img.shape

#Добавляет рамку нулевых пикселей вокруг изображения. ((pad, pad), (pad, pad), (0, 0)) означает: pad сверху/снизу и слева/справа для высоты и ширины, без padding для каналов. mode='constant' заполняет нулями.
    padded = np.pad(img, ((pad, pad), (pad, pad), (0, 0)), mode='constant')

#Создаёт результирующий массив той же формы и типа, что и входное изображение, заполненный нулями.
    result = np.zeros_like(img)
#Двойной цикл проходит по каждому пикселю результирующего изображения (высота × ширина).
    for i in range(rows):
        for j in range(cols):
            #Извлекает окно размером kernel_size × kernel_size из дополненного изображения. Благодаря padding'у окно всегда нужного размера, даже у границ.
            window = padded[i:i+kernel_size, j:j+kernel_size]
            #Цикл по каналам цвета (R, G, B — всего 3).
            for c in range(channels):
                #Для каждого канала вычисляет медиану всех пикселей в окне и записывает её в результирующий пиксель.
                result[i, j, c] = np.median(window[:, :, c])
    
    return result.astype(np.uint8)

Загрузка изображения

In [4]:
img_bgr = cv2.imread('photo23.png')
if img_bgr is None:
    print("Ошибка: не удалось загрузить photo23.png")
    exit()

## Применение фильтра

In [ ]:
# Применяем медианный фильтр
filtered_3x3 = median_filter(img_bgr, 3)  # Блюр 3x3
filtered_5x5 = median_filter(img_bgr, 5)  # Блюр 5x5

# Конвертация в RGB для отображения
img_rgb = bgr2rgb(img_bgr)
filtered_3x3_rgb = bgr2rgb(filtered_3x3)
filtered_5x5_rgb = bgr2rgb(filtered_5x5)

# РАЗДЕЛЕНИЕ КАНАЛОВ для оригинала и блюров
image_noize = img_rgb        # Оригинал (шумный)
image_blur3 = filtered_3x3_rgb  # После блюра 3x3
image_blur5 = filtered_5x5_rgb  # После блюра 5x5


#cv2.split() — функция OpenCV для разделения многоканального изображения на отдельные каналы.
r_noize, g_noize, b_noize = cv2.split(image_noize)
r_blur3, g_blur3, b_blur3 = cv2.split(image_blur3)
r_blur5, g_blur5, b_blur5 = cv2.split(image_blur5)

# ГРАФИКИ g[56] для сравнения (строка 56, зеленый канал)
row_idx = 56

# Проверка высоты
if image_noize.shape[0] <= row_idx:
    print(f"Ошибка: высота изображения {image_noize.shape[0]} < {row_idx+1}")
    exit()


## Визуализация

In [ ]:

plt.figure(figsize=(18, 10))

# Изображения (верхний ряд)
plt.subplot(2, 4, 1)
plt.imshow(image_noize)
plt.title("Оригинал (image_noize)")
plt.axis('off')

plt.subplot(2, 4, 2)
plt.imshow(image_blur3)
plt.title("MF 3×3")
plt.axis('off')

plt.subplot(2, 4, 3)
plt.imshow(image_blur5)
plt.title("MF 5×5")
plt.axis('off')

#  ГРАФИКИ ПРОФИЛЕЙ g[56] (нижний ряд)
plt.subplot(2, 4, 5)
plt.plot(g_noize[row_idx, :], 'g-', linewidth=2, label='Оригинал', alpha=0.8)
plt.title(f"Оригинал g[{row_idx}]")
plt.xlabel("Столбец")
plt.ylabel("Интенсивность")
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(2, 4, 6)
plt.plot(g_blur3[row_idx, :], 'b-', linewidth=2, label='MF 3x3', alpha=0.8)
plt.title(f"MF 3×3 g[{row_idx}]")
plt.xlabel("Столбец")
plt.ylabel("Интенсивность")
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(2, 4, 7)
plt.plot(g_blur5[row_idx, :], 'r-', linewidth=2, label='MF 5x5', alpha=0.8)
plt.title(f"MF 5×5 g[{row_idx}]")
plt.xlabel("Столбец")
plt.ylabel("Интенсивность")
plt.grid(True, alpha=0.3)
plt.legend()


plt.subplot(2, 4, 4)
plt.axis('off')

plt.tight_layout()
plt.show()



## Фильтр Гаусса

Параметры

In [ ]:
KERNEL_SIZE_1 = 5   
SIGMA_1 = 0.8          

KERNEL_SIZE_2 = 5   
SIGMA_2 = 1.3        

 ядро Гаусса

In [ ]:
def gaussian_kernel(size, sigma):

    if size % 2 == 0:
        raise ValueError("Размер ядра должен быть нечётным")
    
    kernel = np.zeros((size, size))
    center = size // 2
    
    #  Полная формула Гауссианы: G(x,y) = 1/(2πσ²) × exp(-(x²+y²)/(2σ²))
    norm_factor = 1.0 / (2 * np.pi * sigma**2)
    
    for i in range(size):
        for j in range(size):
            x, y = i - center, j - center
            # Теоретически правильная формула (сумма ≈ 1)
            kernel[i, j] = norm_factor * np.exp(-(x**2 + y**2) / (2 * sigma**2))
    
    return kernel


Фильтр Гаусса

In [ ]:
def gaussian_filter(img, kernel_size=5, sigma=1.0):
    kernel = gaussian_kernel(kernel_size, sigma)
    pad_h, pad_w = kernel_size // 2, kernel_size // 2
    rows, cols, channels = img.shape
    
    padded = np.pad(img, ((pad_h, pad_h), (pad_w, pad_w), (0, 0)), mode='constant')
    result = np.zeros_like(img)
    
    for i in range(rows):
        for j in range(cols):
            window = padded[i:i+kernel_size, j:j+kernel_size]
            for c in range(channels):
                result[i, j, c] = np.sum(window[:, :, c] * kernel)
    
    return np.clip(result, 0, 255).astype(np.uint8)


## Загрузка изображения 

In [ ]:
img_bgr = cv2.imread('photo22.jpg')
if img_bgr is None:
    print("Ошибка: не удалось загрузить photo22.jpg")
    exit()

## Применение фильтра Гаусса

In [ ]:
kernel1 = gaussian_kernel(KERNEL_SIZE_1, SIGMA_1)
kernel2 = gaussian_kernel(KERNEL_SIZE_2, SIGMA_2)
img1 = gaussian_filter(img_bgr, KERNEL_SIZE_1, SIGMA_1)
img2 = gaussian_filter(img_bgr, KERNEL_SIZE_2, SIGMA_2)

## Визуализация и построение тепловых карт

In [ ]:
# ─── ВИЗУАЛИЗАЦИЯ ───
img_rgb = bgr2rgb(img_bgr)
img1_rgb = bgr2rgb(img1)
img2_rgb = bgr2rgb(img2)

plt.figure(figsize=(15, 5))
plt.subplot(131); plt.imshow(img_rgb); plt.title("Оригинал"); plt.axis('off')
plt.subplot(132); plt.imshow(img1_rgb); plt.title(f"{KERNEL_SIZE_1}×{KERNEL_SIZE_1} (σ={SIGMA_1})"); plt.axis('off')
plt.subplot(133); plt.imshow(img2_rgb); plt.title(f"{KERNEL_SIZE_2}×{KERNEL_SIZE_2} (σ={SIGMA_2})"); plt.axis('off')
plt.tight_layout(); plt.show()

# Тепловые карты 
fig = plt.figure(figsize=(16, 7))

def kernel_grid(size):
    coords = np.arange(size) - (size-1)//2
    return np.meshgrid(coords, coords)

X1, Y1 = kernel_grid(KERNEL_SIZE_1)
ax1 = fig.add_subplot(221, projection='3d')
surf1 = ax1.plot_surface(X1, Y1, kernel1, cmap='viridis', alpha=0.8)
ax1.set_title(f'{KERNEL_SIZE_1}×{KERNEL_SIZE_1} 3D σ={SIGMA_1}')
fig.colorbar(surf1, ax=ax1, shrink=0.6)

ax2 = fig.add_subplot(222)
im2 = ax2.imshow(kernel1, cmap='hot', interpolation='none')
ax2.set_title(f'{KERNEL_SIZE_1}×{KERNEL_SIZE_1} Тепловая σ={SIGMA_1}')
plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
ax2.set_xticks([]); ax2.set_yticks([])

X2, Y2 = kernel_grid(KERNEL_SIZE_2)
ax3 = fig.add_subplot(223, projection='3d')
surf2 = ax3.plot_surface(X2, Y2, kernel2, cmap='plasma', alpha=0.8)
ax3.set_title(f'{KERNEL_SIZE_2}×{KERNEL_SIZE_2} 3D σ={SIGMA_2}')
fig.colorbar(surf2, ax=ax3, shrink=0.6)

ax4 = fig.add_subplot(224)
im4 = ax4.imshow(kernel2, cmap='hot', interpolation='none')
ax4.set_title(f'{KERNEL_SIZE_2}×{KERNEL_SIZE_2} Тепловая σ={SIGMA_2}')
plt.colorbar(im4, ax=ax4, fraction=0.046, pad=0.04)
ax4.set_xticks([]); ax4.set_yticks([])

plt.tight_layout()
plt.show()

# 📋 ДИАГНОСТИКА ЯДЕР
print(f"{'='*60}")
print(f"Ядро {KERNEL_SIZE_1}×{KERNEL_SIZE_1} (σ={SIGMA_1}):")
print(np.round(kernel1, 3))
print(f"Сумма: {kernel1.sum():.6f} ✓")
print(f"\nЯдро {KERNEL_SIZE_2}×{KERNEL_SIZE_2} (σ={SIGMA_2}):")
print(np.round(kernel2, 3))
print(f"Сумма: {kernel2.sum():.6f} ✓")
print(f"{'='*60}")


# Морфологические операции

Преобразование BGR -> GRAY

In [ ]:
def bgr2gray_manual(img):
    if len(img.shape) != 3 or img.shape[2] != 3:
        raise ValueError("Требуется цветное изображение")
    
    b, g, r = img[:,:,0], img[:,:,1], img[:,:,2]
    gray = 0.299 * r + 0.587 * g + 0.114 * b
    return gray.astype(np.uint8)

## Бинаризация

In [ ]:
def binarize_manual(img_gray, threshold=128):
    if len(img_gray.shape) != 2:
        raise ValueError("Требуется grayscale")
    
    binary = np.zeros_like(img_gray)
    binary[img_gray >= threshold] = 255
    return binary.astype(np.uint8)

## Эрозия

In [ ]:
def erosion_manual(img, kernel_size=5):
    if len(img.shape) != 2:
        raise ValueError("Требуется бинарное изображение (H,W)")
    
    pad = kernel_size // 2
    rows, cols = img.shape
    padded = np.pad(img, ((pad, pad), (pad, pad)), mode='constant')
    result = np.zeros((rows, cols), dtype=np.uint8)
    
    # Стандартный квадратный SE = np.ones((kernel_size, kernel_size))
    for i in range(rows):
        for j in range(cols):
            window = padded[i:i+kernel_size, j:j+kernel_size]
            # Эрозия: пиксель = 255 ТОЛЬКО если ВСЕ пиксели в окне = 255
            if np.all(window == 255):
                result[i, j] = 255
    
    return result


Применение эрозии

In [ ]:
img_gray = bgr2gray_manual(img_bgr)
img_binary = binarize_manual(img_gray, threshold=128)
img_eroded = erosion_manual(img_binary, kernel_size=5)  # ← ОДНА эрозия 5x5

Визуализация

In [ ]:
plt.figure(figsize=(15, 8))
# Этап 1-4
plt.subplot(2, 4, 1)
plt.imshow(bgr2rgb(img_bgr))
plt.title("1. Оригинал (цветное)")
plt.axis('off')

plt.subplot(2, 4, 2)
plt.imshow(img_gray, cmap='gray')
plt.title("2. Grayscale (вручную)")
plt.axis('off')

plt.subplot(2, 4, 3)
plt.imshow(img_binary, cmap='gray')
plt.title("3. Бинаризация T=128 (вручную)")
plt.axis('off')

plt.subplot(2, 4, 4)
plt.imshow(img_eroded, cmap='gray')
plt.title("4. Эрозия 5×5 (вручную)")
plt.axis('off')

# Структурирующий элемент (для понимания)
plt.subplot(2, 4, 5)
se_5x5 = np.ones((5,5), dtype=np.uint8)
plt.imshow(se_5x5, cmap='gray')
plt.title("SE: np.ones((5,5), uint8)")
plt.axis('off')

# Сравнение ДО/ПОСЛЕ
plt.subplot(2, 4, 6)
plt.imshow(img_binary, cmap='gray')
plt.title("ДО эрозии")
plt.axis('off')

plt.subplot(2, 4, 7)
plt.imshow(img_eroded, cmap='gray')
plt.title("ПОСЛЕ эрозии")
plt.axis('off')


## Дилатация

In [ ]:
def dilate_manual(mask, kernel_size=(9,9), iterations=1):

    se = np.ones(kernel_size, dtype=np.uint8)
    result = mask.copy()
    
    for _ in range(iterations):
        se_size = se.shape[0]
        pad = se_size // 2
        rows, cols = result.shape
        
        padded = np.pad(result, ((pad, pad), (pad, pad)), mode='constant')
        new_result = np.zeros((rows, cols), dtype=np.uint8)
        
        for i in range(rows):
            for j in range(cols):
                window = padded[i:i+se_size, j:j+se_size]
                # Дилатация: если ХОТЬ ОДИН пиксель в окне = 255
                if np.max(window) == 255:
                    new_result[i, j] = 255
        
        result = new_result
    
    return result



## Применение дилатации

In [ ]:
# Pipeline
img_gray = bgr2gray_manual(img_bgr)
mask = binarize_manual(img_gray, threshold=128)

#  ДИЛАТАЦИЯ 
mask_dil = dilate_manual(mask, kernel_size=(9,9), iterations=1)  # ← ВАШ ПРИМЕР

# Визуализация БЕЗ гистограмм (2x4)
plt.figure(figsize=(16, 8))

# Этап 1-4
plt.subplot(2, 4, 1)
plt.imshow(bgr2rgb(img_bgr))
plt.title("1. Оригинал (цветное)")
plt.axis('off')

plt.subplot(2, 4, 2)
plt.imshow(img_gray, cmap='gray')
plt.title("2. Grayscale (вручную)")
plt.axis('off')

plt.subplot(2, 4, 3)
plt.imshow(mask, cmap='gray')
plt.title("3. MASK (T=128, вручную)")
plt.axis('off')

plt.subplot(2, 4, 4)
plt.imshow(mask_dil, cmap='gray')
plt.title("4. dilate(9×9, iterations=4) ← ВРУЧНУЮ")
plt.axis('off')

# Структурирующий элемент
plt.subplot(2, 4, 5)
kernel = np.ones((9,9), dtype=np.uint8)
plt.imshow(kernel, cmap='gray')
plt.title("Ядро: np.ones((9,9), uint8)")
plt.axis('off')

# Сравнение ДО/ПОСЛЕ
plt.subplot(2, 4, 6)
plt.imshow(mask, cmap='gray')
plt.title("ДО дилатации")
plt.axis('off')

plt.subplot(2, 4, 7)
plt.imshow(mask_dil, cmap='gray')
plt.title("ПОСЛЕ дилатации (×4)")
plt.axis('off')


# Прочие операции

Преобразование BGR ->GRAY

In [ ]:
def bgr2gray_manual(img):
    if len(img.shape) != 3 or img.shape[2] != 3:
        raise ValueError("Требуется цветное изображение")
    
    b, g, r = img[:,:,0], img[:,:,1], img[:,:,2]
    gray = 0.299 * r + 0.587 * g + 0.114 * b
    return gray.astype(np.uint8)

 ## Пороговая бинаризация (для rgb и grayscale изображения)

Бинаризация Grayscale

In [ ]:
def binarize_grayscale(img_gray, threshold=128):
    if len(img_gray.shape) != 2:
        raise ValueError("Требуется grayscale (H,W)")
    
    binary = np.zeros_like(img_gray, dtype=np.uint8)
    binary[img_gray >= threshold] = 255
    return binary


Бинаризация RGB

In [ ]:
def binarize_rgb(img_rgb, threshold=128):
    if len(img_rgb.shape) != 3 or img_rgb.shape[2] != 3:
        raise ValueError("Требуется RGB (H,W,3)")
    
    mean_rgb = np.mean(img_rgb, axis=2).astype(np.uint8)
    binary = np.zeros((img_rgb.shape[0], img_rgb.shape[1]), dtype=np.uint8)
    binary[mean_rgb >= threshold] = 255
    return binary

Бинаризация по каждому RGB каналу

In [ ]:
def binarize_rgb_channels(img_rgb, thresholds=(128, 128, 128)):
    r_bin = (img_rgb[:,:,0] >= thresholds[0]).astype(np.uint8) * 255
    g_bin = (img_rgb[:,:,1] >= thresholds[1]).astype(np.uint8) * 255  
    b_bin = (img_rgb[:,:,2] >= thresholds[2]).astype(np.uint8) * 255
    return r_bin, g_bin, b_bin


Применение бинаризации

In [ ]:
img_rgb = bgr2rgb(img_bgr)
img_gray = bgr2gray_manual(img_bgr)

# Только нужные бинаризации:
gray_bin_128 = binarize_grayscale(img_gray, 128)  # Только T=128
rgb_bin_mean = binarize_rgb(img_rgb, 128)
r_bin, g_bin, b_bin = binarize_rgb_channels(img_rgb, (128, 128, 128))

Визуализация

In [ ]:

plt.figure(figsize=(16, 8))
plt.subplot(2, 4, 1)
plt.imshow(img_rgb)
plt.title("1. Оригинал RGB")
plt.axis('off')

plt.subplot(2, 4, 2)
plt.imshow(gray_bin_128, cmap='gray')
plt.title("2. Gray T=128")
plt.axis('off')

plt.subplot(2, 4, 3)
plt.imshow(rgb_bin_mean, cmap='gray')
plt.title("3. RGB Среднее T=128")
plt.axis('off')

# Каналы RGB
plt.subplot(2, 4, 5)
plt.imshow(r_bin, cmap='gray')
plt.title("4. R канал T=128")
plt.axis('off')

plt.subplot(2, 4, 6)
plt.imshow(g_bin, cmap='gray')
plt.title("5. G канал T=128")
plt.axis('off')

plt.subplot(2, 4, 7)
plt.imshow(b_bin, cmap='gray')
plt.title("6. B канал T=128")
plt.axis('off')



# Выравнивание гистограммы

In [ ]:
def histogram_equalization_manual(img_gray):
    if len(img_gray.shape) != 2:
        raise ValueError("Требуется grayscale")
    
    rows, cols = img_gray.shape
    L = 256
    
    # 1. Гистограмма
    hist = np.zeros(L, dtype=np.int32)
    for i in range(rows):
        for j in range(cols):
            hist[img_gray[i,j]] += 1
    
    # 2. Кумулятивная функция распределения (CDF)
    cdf = hist.cumsum()
    cdf_normalized = cdf * (L - 1) / (rows * cols)
    cdf_normalized = cdf_normalized.astype(np.uint8)
    
    # 3. Маппинг
    result = np.zeros_like(img_gray)
    for i in range(rows):
        for j in range(cols):
            old_intensity = img_gray[i,j]
            result[i,j] = cdf_normalized[old_intensity]
    
    return result, hist, cdf


Загрузка изображения

In [ ]:
img_bgr = cv2.imread('tigr.png')
if img_bgr is None:
    print("Ошибка: не удалось загрузить photo")
    exit()

Применение выравнивания гистограммы

In [ ]:
img_gray = bgr2gray_manual(img_bgr)
img_eq, hist_original, cdf_original = histogram_equalization_manual(img_gray)

# Гистограмма и CDF для ПОСЛЕ эквализации
hist_eq = np.bincount(img_eq.ravel(), minlength=256)
cdf_eq = hist_eq.cumsum()


Визуализация

In [ ]:

# Визуализация (2x4)
plt.figure(figsize=(18, 10))

# Изображения
plt.subplot(2, 4, 1)
plt.imshow(bgr2rgb(img_bgr))
plt.title("Оригинал (RGB)")
plt.axis('off')

plt.subplot(2, 4, 2)
plt.imshow(img_gray, cmap='gray')
plt.title("Grayscale")
plt.axis('off')

plt.subplot(2, 4, 3)
plt.imshow(img_eq, cmap='gray')
plt.title("Выравненная гистограмма")
plt.axis('off')

# ✅ ГИСТОГРАММЫ ОТДЕЛЬНО
plt.subplot(2, 4, 5)
bins = np.arange(256)
plt.bar(bins, hist_original/img_gray.size, alpha=0.7, color='blue', width=1)
plt.title("Гистограмма ДО ")
plt.xlabel("Интенсивность")
plt.ylabel("Норм. частота")
plt.grid(True, alpha=0.3)

plt.subplot(2, 4, 6)
plt.bar(bins, hist_eq/img_eq.size, alpha=0.7, color='red', width=1)
plt.title("Гистограмма ПОСЛЕ")
plt.xlabel("Интенсивность")
plt.ylabel("Норм. частота")
plt.grid(True, alpha=0.3)

# CDF ОТДЕЛЬНО (ДО)
plt.subplot(2, 4, 7)
plt.plot(bins[:len(cdf_original)], cdf_original/255, 'b-', linewidth=3, 
         drawstyle='steps-post', label='CDF ДО')
plt.plot([0,1], [0,1], 'k--', alpha=0.7, label='Идеальная CDF')
plt.title("Кумулятивная функция\nДО ")
plt.xlabel("Интенсивность")
plt.ylabel("Накопленная частота")
plt.legend()
plt.grid(True, alpha=0.3)

# CDF ОТДЕЛЬНО (ПОСЛЕ)
plt.subplot(2, 4, 8)
plt.plot(bins[:len(cdf_eq)], cdf_eq/255, 'r-', linewidth=3, 
         drawstyle='steps-post', label='CDF ПОСЛЕ')
plt.plot([0,1], [0,1], 'k--', alpha=0.7, label='Идеальная CDF')
plt.title("Кумулятивная функция\nПОСЛЕ ")
plt.xlabel("Интенсивность")
plt.ylabel("Накопленная частота")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


 ## Поворот изображений на угол кратный 90 градусов

Поворот на 90° ПО ЧАСОВОЙ

In [ ]:
def rotate_90_clockwise(img):
    """Поворот на 90° ПО ЧАСОВОЙ"""
    if len(img.shape) == 3:
        return np.rot90(img, k=1)
    else:
        return np.rot90(img, k=1)

Поворот на 90° ПРОТИВ ЧАСОВОЙ

In [ ]:
def rotate_90_counterclockwise(img):
    """Поворот на 90° ПРОТИВ ЧАСОВОЙ"""
    if len(img.shape) == 3:
        return np.rot90(img, k=3)
    else:
        return np.rot90(img, k=3)

Поворот на 180° 

In [ ]:
def rotate_180(img):
    """Поворот на 180°"""
    return np.rot90(img, k=2)


Загрузка изображения 

In [ ]:
img_bgr = cv2.imread('photo22.jpg')
if img_bgr is None:
    print("Ошибка: не удалось загрузить photo")
    exit()

Применение поворота

In [ ]:
img_rgb = bgr2rgb(img_bgr)

img_90_cw = rotate_90_clockwise(img_rgb)
img_180 = rotate_180(img_rgb)
img_90_ccw = rotate_90_counterclockwise(img_rgb)


Визуализация

In [ ]:

print("✓ Оригинал:", img_rgb.shape)
print("✓ 90° CW:", img_90_cw.shape)
print("✓ 180°:", img_180.shape)

# Визуализация (только цветные повороты, 1x4)
plt.figure(figsize=(16, 4))

plt.subplot(1, 4, 1)
plt.imshow(img_rgb)
plt.title("Оригинал")
plt.axis('off')

plt.subplot(1, 4, 2)
plt.imshow(img_90_cw)
plt.title("90° По часовой")
plt.axis('off')

plt.subplot(1, 4, 3)
plt.imshow(img_180)
plt.title("180°")
plt.axis('off')

plt.subplot(1, 4, 4)
plt.imshow(img_90_ccw)
plt.title("90° Против часовой")
plt.axis('off')

plt.tight_layout()
plt.show()
